# Veterinary Notes Analyser

This notebook reads a CSV file containing records relating to possible canine lymphoma cases row by row and identifies cases which match the specific case of **"no longer in remission"**, i.e. cases where the patient was judged to be in remission in a previous assessment and, based on new cytology/histology analysis, may be relapsing. The reason for using an LLM for this analysis is that the veterinary reports that are used to make this judgement are unstructured data, and so require a model capable of reasoning about such data to produce useful output.

The notebook reads a CSV file row by row, sends the **last cell** of each row
to a locally-running [LM Studio](https://lmstudio.ai/) server using the **native
LM Studio REST API** (`/api/v1/chat`), and writes the original row plus the
model's response to a new output CSV.

**Prerequisites**
- LM Studio running locally with a model loaded and the local server started (default port: 1234)
- `requests` Python package installed: `pip install requests`

**Typical usage for this project**  
The last cell of each row is the vet's notes column. The model is asked to judge whether or not the report describes a case where the subject is no longer in remission from a previous lymphoma diagnosis and output only 1 if so and 0 if not. The score is appended
as a new column and written to a new file.

## 1. Configuration

Edit these settings before running the notebook.

In [ ]:
# -- File paths -------------------------------------------------------------
INPUT_CSV  = './ExampleData.csv'   # file to read
OUTPUT_CSV = './ExampleData.out.csv'  # file to write

# -- LM Studio server --------------------------------------------------------
# The SDK connects to localhost:1234 by default; change only if you've altered
# LM Studio's port in its settings.
LM_STUDIO_HOST = 'localhost'
LM_STUDIO_PORT = 1234

# Model identifier: pass None to use whichever model is currently loaded in
# LM Studio, or pass a model string, e.g. 'qwen2.5-7b-instruct'.
MODEL_IDENTIFIER = 'nvidia/nemotron-3-nano-4b'

# -- Prompt ------------------------------------------------------------------
# System prompt sent with every request; the last CSV cell is the user message.
SYSTEM_PROMPT = '''
You are a veterinary pathology assistant. You will be given a cytology or histology report
from a veterinarian. Your task is to identify whether the report indicates that the patient
may have relapsed, i.e. is no longer in remission.

In order to consider the patient relapsed, the report must indicate:
- That the patient has completed a prior course of treatment for lymphoma
- That the patient was considered to be in remission previously
- That the remission status is in question with regard to the sample examined

Here is an example of such a report:
"History: FNA of right prescapular LN. Dog previously diagnosed with large cell lymphoma and is on chemotherapy. Was in complete remission but LNs now mildly enlarged.
Assessing for relapse of lymphoma
Slides examined: 4/4 Cytology description: The sample is highly cellular, mildly hemodilute, and overall adequately preserved, though the density of some of the preparation obscures cytologic detail.
Intermediate to large lymphocytes predominate. The cells have a small amount of medium to deep blue cytoplasm. Nuclei are 1-1.5 times the diameter of a neutrophil, eccentric, round, oval, or indented,
and contain finely stippled chromatin often with single distinct, round or oval nucleoli. Occasional mitotic figures are seen. Small, mature lymphocytes are seen in lower number.
Rare vacuolated macrophages are noted. The background contains scattered basophilic cytoplasmic fragments and lysed cells. No infectious agents are identified.
Interpretation: Lymphoma Comment: The cytology smears support relapse of lymphoma in the sampled node."

Note that in the above report, the vet specifically mentions that they are assessing for relapse of lymphoma,
and mentions relapse of lymphoma in the comment on their diagnosis.

Here is an example of a report that DOES NOT match the criteria:
"Gross Examination: The submission consisted of three 2 x 2 to 7 x 4 x 3 mm samples of liver and an approximately 125 x 30-50 x 8 mm sample of spleen with two 10 mm and 22 x 25 mm nodules. Representative areas of spleen were sectioned.
Histopathology: Sections of liver are characterised by regions of subtle portal fibrosis which may bridge portal tracts and central veins, in association with yellow/brown pigmented and vacuolated macrophages, lymphocytes, plasma cells and a few neutrophils or haemopoietic cells.
The same inflammatory cells infiltrate between nearby hepatocytes, around central veins or near the capsule. There is scattered single cell necrosis of hepatocytes near the limiting plate, or elsewhere in the parenchyma.
Some hepatocytes have rarefied to vacuolated cytoplasm. In sections of spleen, there are areas of apparently increased interstitial stromal cells and smooth muscle trabeculae, along with pigmented macrophages in the red pulp (possibly just condensed/non-congested parenchyma).
The nodular masses are regions of focal expansion by myriad variably sized and sometimes coalescing lymphoid aggregates; this is more prominent in the large mass.
The lymphoid aggregates are scattered within a cellular stroma containing spindle cells, lymphocytes, plasma cells, neutrophils, macrophages and haemopoietic cells.
The lymphoid aggregates sometimes resemble follicles with central germinal centre-like areas surrounded by mixed small to large lymphocytes. Other times, they are sheets of mainly intermediate lymphocytes (nuclei 1.5 x red cell diameter) with distinct nucleoli,
which may surround apparently fading germinal centres (large mass). Those cells contain 0-1 mitotic figures per 400x field. The intervening spindle cells are mostly monomorphic with small elongate nuclei.
Diagnosis: Chronic hepatitis (activity grade: moderate, stage: moderate). Splenic marginal zone lymphoma (22 x 25 mm) and nodular hyperplasia (10 mm).
Comment: Michelle, the pattern of inflammation and fibrosis with single cell necrosis in the liver is suggestive of the syndrome of chronic hepatitis. Chronic hepatitis is often idiopathic, although reported causes include copper storage, anticonvulsant drugs or aflatoxicosis,
and some cases have been associated with leptospirosis or infectious canine hepatitis. I will check a copper stain before finalising this case, although I do not see obvious granules suggestive of copper in hepatocytes.
The lymphoid masses are most consistent with an indolent (low-grade) splenic lymphoma, such as marginal zone lymphoma, alongside lymphoid nodular hyperplasia, although it is possible that both lesions are due to lymphoma.
These differentials are difficult to distinguish on H&E and might be distinguished by further testing (e.g. IHC or clonality), however recent literature suggests that they have similar prognoses, unless perhaps the lymphoma cases are documented to become diffuse and spread beyond
the spleen. Marginal zone lymphoma is a B-cell lymphoma that is often low-grade and indolent; affected animals may have lymphadenopathy and/or a splenic mass, but are otherwise well.
After a period of time, marginal zone lymphoma can become more high-grade (immunoblastic or diffuse large B-cell lymphoma). In one study, marginal zone lymphoma was reported to be chemoresponsive, with long standing complete remission. Splenectomy alone may also be curative.
'Canine Indolent Nodular Lymphoma' Vet Path 2006 43:241-256. 'Clinical characteristics and outcome in dogs with splenic marginal zone lymphoma' JVIM 2013 27:949-954. Vet Path 2018 55(5) 645-653.
Addendum (25/5/23 ): With Rhodanine stain, copper granules are identified within some hepatocytes flanking tracts of fibrosis or in areas of vacuolation, and in some macrophages.
Subjectively, I would qualitatively score this as Grade 2/5 on the Cornell Qualitative Scoring criteria (Ref. Canine Copper-Associated Hepatopathy- Where Are We Now? - ACVIM 2017)"

Note that in this report, although the vet makes a reference to remission in the comments and cites some relevant papers, they make no reference to remission or relapse when commenting on the sample itself.
This excludes the report from meeting the criteria.

To reiterate:
- Some reports will give a diagnosis of lymphoma, or probable lymphoma. This is not sufficient on its
  own to consider the patient to have relapsed, as they must have been in remission previously for
  a relapse diagnosis
- Many reports will make references to remission when citing previous studies or other prior work.
  This DOES NOT on its own indicate that the patient was in remission, the criteria above must hold. 

Reply with a single integer (1 if the patient may no longer be in remission, 0 if not) and nothing else.
Note that most reports will not meet the criteria; only reply with '1' if you are certain that the report does.
'''

# -- Output column name ------------------------------------------------------
OUTPUT_COLUMN_NAME = 'LLM_Relapsed?'

# -- Optional: limit rows for testing (set to None to process all rows) ------
MAX_ROWS = None

print('Configuration loaded.')

## 2. Imports and REST client setup

In [ ]:
import csv
import time
import requests

CHAT_URL = f'http://{LM_STUDIO_HOST}:{LM_STUDIO_PORT}/api/v1/chat'

print(f'Will POST to {CHAT_URL}')

## 3. Helper function — query LM Studio

In [ ]:
def query_lm_studio(text: str, retries: int = 3, retry_delay: float = 2.0) -> str:
    """
    Send `text` to LM Studio via the native REST API and return the model's
    response as a stripped string.

    Uses /api/v1/chat with reasoning='off' so models that support reasoning
    return only the final answer token.
    """
    payload = {
        'model': MODEL_IDENTIFIER,
        'system_prompt': SYSTEM_PROMPT,
        'input': text,
        'temperature': 0,
        'max_output_tokens': 16,
        'reasoning': 'off',
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(CHAT_URL, json=payload, timeout=60)
            response.raise_for_status()
            data = response.json()
            # Native API returns the answer in data['response']
            return data['output'][0]['content'].strip()
        except Exception as exc:
            print(f'  Attempt {attempt}/{retries} failed: {exc}')
            if attempt < retries:
                time.sleep(retry_delay)

    return f'ERROR: all {retries} attempts failed'


print('Helper function defined.')

## 4. Quick connectivity test

Run this cell before processing the full file to confirm LM Studio is reachable, and that it responds correctly for a case with an obvious category.

In [ ]:
test_text = '''
History: FNA of right prescapular LN. Dog previously diagnosed with large cell lymphoma and is on chemotherapy. Was in complete remission but LNs now mildly enlarged.
Assessing for relapse of lymphoma
Slides examined: 4/4 Cytology description: The sample is highly cellular, mildly hemodilute, and overall adequately preserved, though the density of some of the preparation obscures cytologic detail.
Intermediate to large lymphocytes predominate. The cells have a small amount of medium to deep blue cytoplasm. Nuclei are 1-1.5 times the diameter of a neutrophil, eccentric, round, oval, or indented,
and contain finely stippled chromatin often with single distinct, round or oval nucleoli. Occasional mitotic figures are seen. Small, mature lymphocytes are seen in lower number.
Rare vacuolated macrophages are noted. The background contains scattered basophilic cytoplasmic fragments and lysed cells. No infectious agents are identified.
Interpretation: Lymphoma Comment: The cytology smears support relapse of lymphoma in the sampled node.
'''
test_result = query_lm_studio(test_text)

print(f'Test input : {test_text}')
print(f'Test output: {test_result}')

if test_result.startswith('ERROR'):
    print('\nCould not reach LM Studio. Make sure the local server is running.')
else:
    print('\nLM Studio is reachable - ready to process the CSV.')

## 5. Process the CSV

Reads `INPUT_CSV` row by row, queries the model using the **last cell** of each
data row, appends the response as a new column, and writes the result to
`OUTPUT_CSV`.

The header row is written first with the extra column name added, then every
data row follows.

In [ ]:
errors    = []   # collect (row_number, error_message) tuples for review after the run
n_written = 0

# Open both file handles.
# 'newline=""' is the recommended way to open files for csv.writer on all platforms.
with open(INPUT_CSV,  encoding='utf-8', newline='') as csv_in, \
     open(OUTPUT_CSV, 'w', encoding='utf-8', newline='') as csv_out:

    reader = csv.reader(csv_in)
    writer = csv.writer(csv_out)

    for row_n, row in enumerate(reader):

        # -- Header row ------------------------------------------------------
        if row_n == 0:
            writer.writerow(row + [OUTPUT_COLUMN_NAME])
            continue
        
        # -- Skip empty rows -------------------------------------------------
        if not row:
            writer.writerow(row + [''])
            continue

        startTime = time.time()
        # -- For now, we are only interested in category 1 samples -----------
        if row[-2].strip() != '1':
            continue

        # -- Optional row limit for quick tests ------------------------------
        if MAX_ROWS is not None and row_n > MAX_ROWS:
            print(f'Reached MAX_ROWS={MAX_ROWS} - stopping early.')
            break

        # -- Extract the last cell and query the model -----------------------
        last_cell = row[-1].strip()

        if not last_cell:
            llm_response = ''
        else:
            llm_response = query_lm_studio(last_cell)

        if llm_response.startswith('ERROR'):
            errors.append((row_n, llm_response))

        # -- Write augmented row to the output file --------------------------
        writer.writerow(row + [llm_response])
        n_written += 1
        endTime = time.time()

        # -- Progress indicator ----------------------------------------------
        print(f'Row {row_n}: {llm_response} - {endTime - startTime}s')

print(f'\nDone. {n_written} data rows written to {OUTPUT_CSV}')

if errors:
    print(f'\n{len(errors)} errors encountered:')
    for row_n, msg in errors:
        print(f'  Row {row_n}: {msg}')
else:
    print('No errors.')

## 6. Spot-check the output

In [ ]:
PREVIEW_ROWS = 5

with open(OUTPUT_CSV, encoding='utf-8', newline='') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i > PREVIEW_ROWS:
            break
        display_row = [cell[:50] if len(cell) > 50 else cell for cell in row]
        print(f'Row {i:>3}: {display_row}')